# Exercise 6 - Simulating Survey Responses
_By Georg Ahnert_

In this exercise we will look at how LLMs can be used to generate synthetic survey responses. We will also look into how to evaluate them against human survey data.

**You will likely need to run this notebook on the BWUniCluster3.0 or on Google Colab to have enough GPU memory and compute.**

This exercise consists of three parts:
1. Persona simulation
2. Generating closed-ended responses
2. Evaluation against human survey data

### Setup

Follow the instructions from Exercise 3 for LLM setup. Make sure you are using the correct kernel is selected for this notebook.

In [ ]:
%pip install vllm
%pip install scikit-learn

## 1. Persona simulation

Surveys are often simulated _in-silico_ with LLMs on an individual level. We provide the LLM with a description of a persona to predict the survey responses of the same individual.

Here, we will use data from the American National Election Study (ANES) to simulate a U.S. population. This closely follows the work of [Argyle et al. (2023)](https://www.cambridge.org/core/journals/political-analysis/article/out-of-one-many-using-language-models-to-simulate-human-samples/035D7C8A55B237942FB6DBAD7CAA4E49).

First, let's load a subset of the ANES 2016 dataset. For more detail on the ANES, see https://electionstudies.org/data-center/2016-time-series-study/

In [ ]:
import pandas as pd

argyle_df = pd.read_csv('2016_anes_argyle.csv')
argyle_df

As you can see, we have both sociodemographic variables (e.g. age, gender) and attitudes (e.g., ideology), as well as a "ground_truth" column of self-reported vote choice.

Note that some individuals have _item-level missingness_, i.e., did not respond to all questions. The survey also has _person-level missingness_, i.e., some people that have been originally sampled did not respond to the survey at all.

Still, the ANES tries to sample survey participants that can represent the population of eligible voters in the U.S.

We will work with a small, random subsample of this for this exercise.

In [ ]:
sample_df = argyle_df.sample(n=50, random_state=42)
sample_df.head()

There are a couple of commonly used approaches to simulating personas (see also https://arxiv.org/abs/2507.16076).

Let's try a key-value format first:

In [ ]:
personas = []

for id, individual in sample_df.iterrows():
    valid_responses = individual.dropna() # remove item-level missingness from the simulation
    persona_attributes = valid_responses.drop('ground_truth') # remove the column that we want to predict
    personas.append(f"Pretend you are the following person: {persona_attributes.to_dict()}")

print(personas[0])
print(personas[4])

Another way to implement personas would be "interview-style" question-answer pairs:

In [ ]:
personas = []

for id, individual in sample_df.iterrows():
    persona = ''
    if pd.notna(individual.race): # only add existing data
        persona += f"Interviewer: What is your race?\nInterviewee: I am {individual.race.title()}.\n"
    if pd.notna(individual.age):
        persona += f"Interviewer: How old are you?\nInterviewee: I am {int(individual.age)} years old.\n"
    if pd.notna(individual.political_interest):
        persona += f"Interviewer: Are you interested in politics?\nInterviewee: I am {individual.political_interest} interested in politics.\n"
    personas.append(persona)

print(personas[0])
print(personas[4])

### Task 1

[Argyle et al. (2023)](https://www.cambridge.org/core/journals/political-analysis/article/out-of-one-many-using-language-models-to-simulate-human-samples/035D7C8A55B237942FB6DBAD7CAA4E49) used "biography" prompt format. Have a look at the supplementary material of their paper and re-implement the prompt format of study 2 using the data stored in `sample_df`.

## 2. Generating closed-ended responses

Now that we have personas implemented, let's actually try to predict survey answers. First, let's set up the model. We'll use Qwen 3 4B here, since it's a bit better at instruction following than OLMo 2 1B.

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen3-4B-Instruct-2507")

Let's use the same prompt as Argyle et al. (2023) to try to predict vote choice:

In [ ]:
from transformers import set_seed

set_seed(42)

batch_inputs = [
    [
        {
            "role": "user",
            "content": f"{persona}\nIn the 2016 presidential election, I voted for"
        }
    ]
    for persona in personas[:2]
]
    
outputs = pipe(batch_inputs, max_new_tokens=100)

In [ ]:
for conversation in outputs:
    reply = conversation[0]['generated_text'][-1]['content']
    print('\n---\n')
    print(reply)

We observe that the model is trained to produce open-ended text instead of responding with a closed-ended survey response. But maybe some instruction in the system prompt can help.

In [ ]:
batch_inputs = [
    [
        {
            "role": "system",
            "content": "You are a political scientist that predicts survey responses. " + \
                       "The possible answer options are: ['Clinton', 'Trump', 'Non-voter']. " + \
                        "Only respond with the most likely answer option. Do not produce any additional text!"
        },
        {
            "role": "user",
            "content": f"{persona}\nIn the 2016 presidential election, I voted for"
        }
    ]
    for persona in personas
]
    
single_outputs = pipe(batch_inputs, max_new_tokens=100)

In [ ]:
for conversation in single_outputs[:10]:
    reply = conversation[0]['generated_text'][-1]['content']
    print(reply)

### Task 2

An alternative to a single label is to let the model generate a probability distribution across the answer options for each individual. See for instance [Meister et al. (2024)](https://aclanthology.org/2025.naacl-long.2/) or [Ahnert et al. (2025)](https://arxiv.org/abs/2510.11586). Implement this "Verbalized Distribution" method using the following JSON format:

```
{
    "Clinton": <probability>,
    "Trump": <probability>,
    "Non-voter": <probability>
}
```

## 3. Evaluation against human survey data

Now, let's see how well we were actually able to predict survey responses. Let's start with an individual-level evaluation first:

In [ ]:
from sklearn.metrics import classification_report

predicted_labels = [conversation[0]['generated_text'][-1]['content'] for conversation in single_outputs]
predicted_labels[:10]

In [ ]:
true_labels = sample_df['ground_truth'].to_list()
true_labels[:10]

In [ ]:
print(classification_report(
    true_labels,
    predicted_labels,
    labels=['Trump', 'Clinton', 'Non-voter'] # ignore additional labels that the LLM hallucinated
))

We are reasonably good with Trump and Clinton voters, but predicting Non-voters is hard.

Another way to look at evaluation is a distribution match. Do we get the response distributions in subpopulations correct? Let's have a look at the people who do and do not go to church.

In [ ]:
result_df = sample_df.copy()
result_df['prediction'] = predicted_labels

gt_distr = result_df.groupby('church_goer')['ground_truth'].value_counts(normalize=True)
gt_distr

In [ ]:
pred_distr = result_df.groupby('church_goer')['prediction'].value_counts(normalize=True)
pred_distr # NOTE that Non-voters have never been predicted for the "attend church" group!

To measure similarity between these two distributions, we can calculate [total variation distance](https://en.wikipedia.org/wiki/Total_variation_distance_of_probability_measures) and [Jensen-Shannon Divergence](https://en.wikipedia.org/wiki/Jensen%E2%80%93Shannon_divergence) as follows:

In [ ]:
import numpy as np
from scipy.spatial.distance import jensenshannon

# Make sure that all labels (i.e., also Non-voters) are found in all groups in the same order
index = ['Trump', 'Non-voter', 'Clinton']

pred_labels = pred_distr.reset_index(level=0)
gt_labels = gt_distr.reset_index(level=0)
display(gt_labels)

for subpopulation in ['attend church', 'do not attend church']:
    subpop_pred = pred_labels[pred_labels.church_goer == subpopulation].reindex(index=index).fillna(0)
    subpop_gt = gt_labels[gt_labels.church_goer == subpopulation].reindex(index=index).fillna(0)
    
    tv_distance = 0.5 * np.abs(subpop_gt.proportion.to_numpy() - subpop_pred.proportion.to_numpy()).sum()
    print('---')
    print(f"Total Variation Distance for '{subpopulation}': {tv_distance:.3f} (lower is better)")

    js_divergence = jensenshannon(subpop_gt.proportion.to_numpy(), subpop_pred.proportion.to_numpy()) ** 2 # returns sqrt(JSD) by default
    print(f"Jensen-Shannon Divergence for '{subpopulation}': {js_divergence:.3f} (lower is better)")

### Task 3

How does the Verbalized Distribution output from Task 2 evaluate on subpopulation-level distributions?

1. Parse the JSON responses from Task 2
2. Calculate the mean response distribution in each subpopulation
3. Calculate Total Variation Distance and Jensen-Shannon Divergence for both subpopulations

### Task 4 (Optional)

Re-run the experiments with different persona prompt formats and answer scales. You can, for instance, try a "multiple choice question" scale that uses ["A", "B", "C"] as the possible answer options, or you can reverse the order in which the answer options are presented. What do you observe in terms of individual-level macro avg. F1-score and in terms of subpopulation level Total Variation Distance? How far to the outcomes differ between the simulation specifications? Which specification yields the best results?